# Tutorial 3a: Optimization Basics

This tutorial introduces the **Optiland optimization subsystem**. By the end you will know how to:

* Build an `OptimizationProblem` with operands and variables
* Call `minimize()` with automatic method selection
* Interpret the `OptimizationResult` object
* Understand the **mutation contract** — that `minimize()` updates the optic in-place

Later tutorials cover method selection (3b) and advanced DLS/LM tuning (3c).

## 1. Imports

In [ ]:
import copy
import numpy as np

from optiland import optic
from optiland import analysis
from optiland.optimization import OptimizationProblem, minimize

## 2. Build the starting lens

We define a simple **plano-convex singlet** made from N-BK7 glass. The system has a 10 mm entrance pupil diameter (EPD), three wavelengths spanning the visible spectrum, and two field angles (on-axis and 70% of max). The image distance is deliberately set short so the optimizer has real work to do.

In [ ]:
lens = optic.Optic()

# Object plane (at infinity)
lens.surfaces.add(index=0, thickness=np.inf)

# Lens front surface (stop coincident with glass)
lens.surfaces.add(index=1, thickness=7, radius=50, material='N-BK7', is_stop=True)

# Lens rear surface
lens.surfaces.add(index=2, thickness=45, radius=-50)

# Image plane
lens.surfaces.add(index=3)

# Aperture
lens.set_aperture(aperture_type='EPD', value=10)

# Fields
lens.fields.set_type('angle')
lens.fields.add(y=0.0)   # on-axis
lens.fields.add(y=0.7)   # 70% of max field

# Wavelengths (F, d, C lines)
lens.wavelengths.add(value=0.4861)               # F
lens.wavelengths.add(value=0.5876, is_primary=True)  # d (primary)
lens.wavelengths.add(value=0.6563)               # C

lens.update_paraxial()
lens.draw()

## 3. Define the optimization problem

An `OptimizationProblem` is a container that holds:

* **Operands** — quantities to minimise (e.g. spot size, focal length error). Each operand has a *target*, a *weight*, and an *input_data* dict that tells the operand how to evaluate itself against a specific `Optic` instance.
* **Variables** — lens parameters the optimizer is allowed to change (radius, thickness, conic, asphere coefficients, etc.).

Here we add:

| Operand | Target | Purpose |
|---------|--------|---------|
| `rms_spot_size` | 0 | Minimise on-axis blur |
| `f2` | 100 mm | Keep focal length at 100 mm |

And three variables: the two surface radii and the image distance.

In [ ]:
problem = OptimizationProblem()

# Operand 1 — on-axis RMS spot size
spot_input = {
    'optic': lens,
    'surface_number': -1,   # image surface
    'Hx': 0,
    'Hy': 0,
    'num_rays': 5,
    'wavelength': 0.5876,
    'distribution': 'hexapolar',
}
problem.add_operand('rms_spot_size', target=0, weight=1, input_data=spot_input)

# Operand 2 — focal length
problem.add_operand('f2', target=100, weight=1, input_data={'optic': lens})

# Variables
problem.add_variable(lens, 'radius', surface_number=1)
problem.add_variable(lens, 'radius', surface_number=2)
problem.add_variable(lens, 'thickness', surface_number=2, min_val=30, max_val=80)

# Inspect the problem before optimizing
problem.info()

## 4. Run with automatic method selection

Calling `minimize(problem)` without specifying a method uses `method='auto'`. Optiland inspects the problem — number of variables, operand types, presence of bounds — and picks an appropriate algorithm at runtime. The chosen method is recorded in `result.method`, and `result.resolved_from` will be `'auto'` to signal that the selection was automatic.

**Mutation contract:** `minimize()` mutates the `Optic` object that was referenced inside `input_data`. We take a deep copy *before* calling `minimize()` so we can compare starting and final lens parameters.

In [ ]:
# Snapshot the starting lens before any mutation
starting_lens = copy.deepcopy(lens)

# Run with automatic method selection
result = minimize(problem)  # method='auto' is the default

print(result)

## 5. Reading the OptimizationResult

The `result` object exposes a rich set of fields so you can understand what happened without re-running the optimization.

| Field | Type | Meaning |
|-------|------|---------|
| `result.method` | `str` | Algorithm that was actually used |
| `result.resolved_from` | `str \| None` | `'auto'` if auto-selected, `None` if explicit |
| `result.value` | `float` | Final scalar merit function value |
| `result.improvement_pct` | `float` | Percent improvement from starting merit |
| `result.success` | `bool` | Whether the solver declared convergence |
| `result.stop_reason` | `str` | Human-readable stopping criterion |
| `result.x` | `np.ndarray` | Final parameter vector |
| `result.wall_time_s` | `float` | Elapsed wall time in seconds |

In [ ]:
print(f'Method used:      {result.method}')
print(f'Resolved from:    {result.resolved_from}')
print(f'Final merit:      {result.value:.6f}')
print(f'Improvement:      {result.improvement_pct:.1f}%')
print(f'Converged:        {result.success}')
print(f'Stopped because:  {result.stop_reason}')
print(f'Wall time:        {result.wall_time_s:.3f} s')
print(f'Final parameters: {result.x}')

## 6. The mutation contract

`minimize()` modifies the optic **in-place**. After the call, `lens` holds the optimized parameters; `starting_lens` (the deep copy we made before calling `minimize()`) is untouched.

This is intentional: it lets you chain multiple optimization passes without rebuilding the problem from scratch. If you want to compare or revert, always `copy.deepcopy(lens)` *before* you call `minimize()`.

Below we print the surface radii and image distance for both lenses:

In [ ]:
def print_params(label, optic_obj):
    r1 = optic_obj.surfaces[1].geometry.c
    r1 = 1 / r1 if r1 != 0 else float('inf')
    r2 = optic_obj.surfaces[2].geometry.c
    r2 = 1 / r2 if r2 != 0 else float('inf')
    t2 = optic_obj.surfaces[2].geometry.cs.t
    print(f'{label}:')
    print(f'  R1 = {r1:+.4f} mm')
    print(f'  R2 = {r2:+.4f} mm')
    print(f'  Image distance = {t2:.4f} mm')

print_params('Starting lens', starting_lens)
print()
print_params('Optimized lens', lens)

## 7. Visualize the optimized lens

Draw the optimized lens and generate a spot diagram to confirm the improvement visually.

In [ ]:
lens.draw()

In [ ]:
spot = analysis.SpotDiagram(lens)
spot.view()

## 8. Post-optimization problem info

`problem.info()` re-evaluates every operand against the *current* state of the optic and prints a summary table. This is the quickest way to see which operands are still contributing most to the merit function.

In [ ]:
problem.info()

## Summary

You have learned:

* How to construct an `OptimizationProblem` with operands and bounded variables
* How to call `minimize()` with `method='auto'` and let Optiland select the algorithm
* What each field of `OptimizationResult` means
* That `minimize()` mutates the optic in-place — always deep-copy first if you need to preserve the starting state

**Next:** [Tutorial 3b — Choosing a Method](Tutorial_3b_Choosing_a_Method.ipynb) explains the method families and when to pick each one.